# 36 - Compare old (production-based) vs new (LLM-validated pooled) gold standard

For the 5 pilot queries judged in notebook 35, recomputes Precision@k/Recall@k/NDCG@k against the new `final_gold_labels.json` (0/1/2 relevance grades from judge-agreement plus your manual review), and puts it side by side with the OLD metrics already computed against the frozen production ground truth for the same 5 queries.

This is the actual test of whether building a non-circular gold standard changes anything: does 'best method' still look the same?

Two relevance thresholds are reported separately: `relaxed` (gold_label >= 1, includes partial matches) and `strict` (gold_label == 2, highly relevant only).

Also includes a judge-calibration check: across the 70 disagreements from notebook 35, did OpenAI or Claude agree with your final call more often?

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("result/36_new_gold_standard_comparison")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

gold = pd.DataFrame(json.load(open("result/35_llm_judge_ensemble/final_gold_labels.json")))
PILOT_QUERY_IDS = sorted(gold["query_id"].unique().tolist())
K_VALUES = [5, 10, 20, 50]

print(f"Pilot queries: {PILOT_QUERY_IDS}")
print(gold.groupby("query_id").size().rename("gold_pool_size"))

Pilot queries: [1, 2, 3, 4, 5]
query_id
1    90
2    86
3    70
4    91
5    82
Name: gold_pool_size, dtype: int64


In [2]:
def new_relevant_sets(query_id, threshold):
    sub = gold[gold["query_id"] == query_id]
    return set(sub[sub["gold_label"] >= threshold]["domain"])


def new_graded_scores(query_id):
    sub = gold[gold["query_id"] == query_id]
    return dict(zip(sub["domain"], sub["gold_label"]))


def precision_at_k(retrieved, relevant_set, k):
    return sum(1 for d in retrieved[:k] if d in relevant_set) / k if k else 0.0


def recall_at_k(retrieved, relevant_set, k):
    if not relevant_set:
        return np.nan
    return sum(1 for d in retrieved[:k] if d in relevant_set) / len(relevant_set)


def ndcg_at_k(retrieved, graded_scores, k):
    dcg = sum(graded_scores.get(d, 0) / np.log2(i + 2) for i, d in enumerate(retrieved[:k]))
    ideal = sorted(graded_scores.values(), reverse=True)[:k]
    idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal))
    return dcg / idcg if idcg else 0.0

In [3]:
def load_ranked(method):
    if method == "bm25":
        df = pd.read_csv("result/01_baseline_bm25/bm25_results.csv")
    elif method == "gte_large":
        df = pd.read_csv("result/17_baseline_gte_large/gte_large_results.csv")
    elif method == "colbert":
        df = pd.read_csv("result/27_baseline_colbert/colbert_results.csv")
    elif method == "learned_fusion":
        df = pd.read_csv("result/30_learned_fusion_ranker/scored_candidates.csv")
        df["rank"] = df.groupby("query_id")["score_gbdt"].rank(ascending=False, method="first")
    return df[df["query_id"].isin(PILOT_QUERY_IDS)][["query_id", "domain", "rank"]]


def load_old_eval(method):
    paths = {
        "bm25": "result/01_baseline_bm25/evaluation_bm25.csv",
        "gte_large": "result/17_baseline_gte_large/evaluation_gte_large.csv",
        "colbert": "result/27_baseline_colbert/evaluation_colbert.csv",
        "learned_fusion": "result/30_learned_fusion_ranker/evaluation_gbdt.csv",
    }
    df = pd.read_csv(paths[method])
    return df[(df["query_id"].isin(PILOT_QUERY_IDS)) & (df["k"].isin(K_VALUES))]


METHODS = ["bm25", "gte_large", "colbert", "learned_fusion"]
print(f"Comparing: {METHODS}")

Comparing: ['bm25', 'gte_large', 'colbert', 'learned_fusion']


In [4]:
rows = []
for method in METHODS:
    ranked = load_ranked(method)
    old_eval = load_old_eval(method)

    for query_id in PILOT_QUERY_IDS:
        retrieved = ranked[ranked["query_id"] == query_id].sort_values("rank")["domain"].tolist()
        relaxed_set = new_relevant_sets(query_id, threshold=1)
        strict_set = new_relevant_sets(query_id, threshold=2)
        graded = new_graded_scores(query_id)

        for k in K_VALUES:
            old_row = old_eval[(old_eval["query_id"] == query_id) & (old_eval["k"] == k)]
            rows.append({
                "method": method, "query_id": query_id, "k": k,
                "old_precision": old_row["precision"].iloc[0] if len(old_row) else np.nan,
                "old_recall": old_row["recall"].iloc[0] if len(old_row) else np.nan,
                "old_ndcg": old_row["ndcg"].iloc[0] if len(old_row) else np.nan,
                "new_precision_relaxed": precision_at_k(retrieved, relaxed_set, k),
                "new_recall_relaxed": recall_at_k(retrieved, relaxed_set, k),
                "new_precision_strict": precision_at_k(retrieved, strict_set, k),
                "new_recall_strict": recall_at_k(retrieved, strict_set, k),
                "new_ndcg": ndcg_at_k(retrieved, graded, k),
            })

comparison_df = pd.DataFrame(rows)
comparison_df.to_csv(OUTPUT_DIR / "old_vs_new_comparison.csv", index=False)

summary = comparison_df.groupby(["method", "k"])[[
    "old_precision", "new_precision_relaxed", "new_precision_strict",
    "old_recall", "new_recall_relaxed", "new_recall_strict",
    "old_ndcg", "new_ndcg",
]].mean().round(3)
print(summary)
print(f"\nSaved per-query detail -> {OUTPUT_DIR / 'old_vs_new_comparison.csv'}")

                   old_precision  new_precision_relaxed  new_precision_strict  \
method         k                                                                
bm25           5             NaN                  0.880                 0.360   
               10          0.800                  0.920                 0.440   
               20            NaN                  0.910                 0.410   
               50          0.772                  0.476                 0.244   
colbert        5             NaN                  1.000                 0.680   
               10          0.760                  0.980                 0.720   
               20            NaN                  0.950                 0.720   
               50          0.856                  0.564                 0.412   
gte_large      5             NaN                  1.000                 0.680   
               10          0.800                  0.960                 0.720   
               20           

## Judge calibration check

Across the 70 disagreements, did OpenAI or Claude agree with your final call more often?

In [5]:
review = pd.read_csv("result/35_llm_judge_ensemble/review_queue.csv")
review = review[review["your_label"].astype(str).str.upper() != "SKIP"].copy()
review["your_label"] = review["your_label"].astype(int)

openai_agree = (review["openai_label"] == review["your_label"]).mean()
claude_agree = (review["claude_label"] == review["your_label"]).mean()

print(f"Cases reviewed: {len(review)}")
print(f"OpenAI matched your final call : {openai_agree:.1%}")
print(f"Claude matched your final call : {claude_agree:.1%}")

Cases reviewed: 59
OpenAI matched your final call : 13.6%
Claude matched your final call : 84.7%
